# 05. Streaming: muitas métricas, guardrails e correção múltipla

**A decisão:** o time de recomendação treinou um novo algoritmo e quer saber se
pode substituir o que está em produção. A pergunta parece simples, mas o problema
não é uma métrica, são várias ao mesmo tempo: o tempo assistido (a métrica
primária), o número de sessões, a taxa de conclusão e um guardrail de buffering
(que não pode piorar). Lançar significa acertar todas essas leituras de uma vez,
sem se enganar com o acaso.

Este capítulo fecha a trilha de casos de uso. O aprendizado central é sobre
disciplina estatística: quando você olha para muitas métricas, a chance de
encontrar um "ganho" que é puro ruído cresce rápido, e um cientista sênior corrige
isso antes de comemorar. Vamos medir essa inflação, aplicar uma correção de
múltiplos testes e, o mais importante, aprender a ler um resultado nulo com
honestidade.

Fundamentos por trás deste capítulo:
[IV. Inferência](../../../docs/pt-br/teoria/04-inferencia.md) (tópico 16) e
[V. Diagnósticos](../../../docs/pt-br/teoria/05-diagnosticos.md) (tópico 21).


## O reflexo natural (e por que ele não basta)

O instinto é direto: rodar um teste em cada uma das quatro métricas, cada um a
`alpha = 0,05`, e comemorar qualquer vencedor. Parece inofensivo, cada teste
sozinho está calibrado para errar só 5% das vezes sob a hipótese nula.

O problema aparece quando você olha para a **família** de testes de uma vez. Com
várias hipóteses nulas independentes, a chance de **pelo menos um** falso positivo
não fica em 5%: ela infla bem acima disso. É esse o motivo pelo qual precisamos
corrigir para múltiplos testes. Antes de corrigir, vamos medir o tamanho do
problema.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "streaming_metrics.csv")
print(df.shape)
df.head()

## O erro por família infla

Com `K` hipóteses nulas independentes, cada uma testada a `alpha`, a probabilidade
de **pelo menos um** falso positivo é `1 - (1 - alpha)^K`. Para `K = 4` a
`alpha = 0,05`, isso dá cerca de 18,5%: quase um em cada cinco "ganhos" seria puro
ruído. A fórmula é analítica, mas vale confirmar por simulação para ter certeza de
que o número não é um artefato de raciocínio.


In [ ]:
# Ana's reflex is to test all four metrics at alpha = 0.05 and celebrate any winner.
# But with several independent nulls, the chance of at least one false positive
# balloons well above 5%. That is why we must correct for multiple testing.
K, alpha = 4, 0.05
print(f"analytic P(>=1 false positive), {K} independent nulls: {1 - (1 - alpha) ** K:.1%}")

rng = np.random.default_rng(505)
sims = 20000
any_fp = sum(bool((rng.random(K) < alpha).any()) for _ in range(sims))
print(f"simulated                                        : {any_fp / sims:.1%}")

## Uma alocação, várias métricas

Um ponto sutil que separa quem entende de experimentação de quem só roda testes:
não existem quatro experimentos aqui, existe **um só**. Os mesmos usuários foram
randomizados uma única vez, e essa mesma alocação vale para todas as métricas.
Randomizar de novo para cada métrica seria misturar experimentos diferentes.

Então fazemos a atribuição uma vez com `CRD` e a reutilizamos. Para cada métrica,
montamos o resultado observado (o valor de tratamento onde `t == 1`, o de controle
caso contrário) e estimamos o efeito com `NeymanCI`.


In [ ]:
from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

design = CRD(p=0.5, seed=505)
base = design.randomize(df[["plan"]].copy())
t = base.data_[base.treatment_col_].to_numpy()

metrics = ["watch_time", "sessions", "completion", "buffering"]
results = {}
for m in metrics:
    data = base.data_.copy()
    data[m] = np.where(t == 1, df[f"{m}_y1"].to_numpy(), df[f"{m}_y0"].to_numpy())
    a = CRDAssignment(data=data, treatment_col=base.treatment_col_, design=design, seed=505)
    results[m] = NeymanCI(estimator=DifferenceInMeans(outcome_col=m)).fit(a).estimate()

{m: round(r.p_value, 4) for m, r in results.items()}

## Correção de múltiplos testes (FWER, Holm)

Com quatro métricas na família, controlamos a **taxa de erro por família** (FWER,
family-wise error rate) com o método de Holm. Holm é um ajuste sequencial dos
p-valores: ele garante que a probabilidade de **qualquer** falso positivo na
família toda fique em `alpha`, e é mais poderoso que o clássico Bonferroni sem
abrir mão dessa garantia rígida.

O `ExperimentComparison` aplica a correção sobre a família e devolve uma tabela
pronta com o p-valor original e o corrigido, além de um forest plot para visualizar
os efeitos e seus intervalos lado a lado.


In [ ]:
from skxperiments.pipeline import ExperimentComparison
from skxperiments.reporting import plot_forest

comparison = ExperimentComparison(correction="holm", alpha=0.05).run(results)
display_cols = ["experiment", "ate", "p_value", "p_value_corrected", "significant"]
print(comparison.table[display_cols].round(4).to_string(index=False))

ax = plot_forest(comparison)
ax.set_title("Effect by metric (Holm-corrected)")
ax.figure

## Lendo o nulo e o guardrail (limites honestos)

Só o tempo assistido sobrevive à correção. Aqui está a leitura honesta que um
cientista sênior insiste em fazer, e que um tutorial de API costuma esconder.

**Sessões e conclusão "não significativas" não é "provado que não têm efeito".**
Ausência de evidência não é evidência de ausência: essas métricas estão apenas com
poder estatístico baixo para descartar efeitos pequenos. Elas ficam inconclusivas,
não zeradas.

**O guardrail de buffering "não sinalizado" não é o mesmo que "certificado
seguro".** Um guardrail pede raciocínio de **não inferioridade** (provar que ele não
piorou de forma relevante), e um teste bilateral "não significativo" não estabelece
isso sozinho. A `skxperiments` v1 **não tem** teste dedicado de não inferioridade ou
equivalência, então trate um guardrail limpo como "sem alarme", não como "provado
inofensivo".

**Uma escolha que merece nome:** Holm controla o erro por família (rígido, qualquer
falso positivo conta) e é o certo para um portão de lançamento; o Benjamini-Hochberg
controla a taxa de falsas descobertas (FDR, mais frouxo) e é melhor para varreduras
exploratórias de muitas métricas. A escolha depende do custo de errar.


## Pipeline e relatório na métrica primária

Para fechar, rodamos o `ExperimentPipeline` na métrica primária. Além da inferência,
ele executa o `SRMTest` automaticamente (o diagnóstico de sample-ratio mismatch que
vimos nos capítulos anteriores), garantindo que a alocação 50/50 não quebrou.
Depois geramos um `ExperimentReport` em HTML, pronto para compartilhar com quem toma
a decisão.


In [ ]:
from skxperiments.diagnostics import SRMTest
from skxperiments.pipeline import ExperimentPipeline

data = base.data_.copy()
data["watch_time"] = np.where(t == 1, df["watch_time_y1"].to_numpy(), df["watch_time_y0"].to_numpy())
primary = CRDAssignment(data=data, treatment_col=base.treatment_col_, design=design, seed=505)

pipeline_result = ExperimentPipeline(
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="watch_time")),
    diagnostics=[SRMTest()],
).run(primary)
print(f"watch_time ATE={pipeline_result.results.ate:.3f}, "
      f"p={pipeline_result.results.p_value:.4f}, flagged={pipeline_result.flagged}")

In [ ]:
from IPython.display import HTML

from skxperiments.reporting import ExperimentReport

report = ExperimentReport(pipeline_result, title="Streaming recommender - watch time")
HTML(report.to_html())

## Decisão, limites e fechamento

O efeito verdadeiro no tempo assistido é de **+3 minutos**, o único real, e ele
sobrevive à correção de Holm; o guardrail não levanta alarme. **Recomendação:
lançar**, monitorando o buffering de perto e tratando as métricas secundárias como
inconclusivas, não nulas. Essa disciplina, distinguir "não sabemos" de "não tem
efeito", é o que separa uma decisão defensável de um wishful thinking com p-valor.

Limites deste capítulo, ditos com honestidade:
1. Sem teste de não inferioridade ou equivalência para guardrails na v1: um guardrail
   limpo é "sem alarme", não prova de segurança.
2. Holm assume que a família é exatamente estas quatro métricas: pré-registre a
   família de métricas antes de olhar os dados, ou a garantia de FWER se dilui.
3. Nenhuma redução de variância por métrica foi aplicada: CUPED apertaria as
   leituras e daria mais poder para as secundárias.

Com isso, encerramos a trilha de casos de uso. Para os fundamentos por trás de cada
peça, veja a [série de teoria](../../../docs/pt-br/teoria/01-fundamentos.md).
